In [1]:
import pandas as pd
import numpy as np

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [2]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)
data_gdynia.head()

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking
id,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,hotel,NaN,Hotel,7,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27
2,890729731,Świętego Piotra,6D,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27
3,890292040,Węglowa,22,NaN,centrum handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27
4,890729732,Świętego Piotra,6,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27
5,136885205,Tadeusza Wendy,15,NaN,centrum medyczne,NaN,Centrum Medyczne,6,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,2,2026-09-27


In [3]:
data_gdynia_long = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long.head()

,addr_street,ground_floor_function
id,,
1,Świętego Piotra,hotel
2,Świętego Piotra,mieszkaniowa wielorodzinna
3,Węglowa,centrum handlowe
4,Świętego Piotra,mieszkaniowa wielorodzinna
5,Tadeusza Wendy,centrum medyczne


1.

In [ ]:
build_per_street = data_gdynia.groupby("addr_street", as_index=False).agg(
    count_build=("osm_id", "size")
)

z_left = pd.merge(data_gdynia, build_per_street, on="addr_street", how="left")
z_inner = pd.merge(data_gdynia, build_per_street, on="addr_street", how="inner")
print(f"z left_len: {len(z_left)}")
print(f"z inner_len: {len(z_inner)}")


z left_len: 1105
z inner_len: 837


In [5]:
assert len(z_left) - len(z_inner) == data_gdynia["addr_street"].isna().sum()

2.

In [6]:
big_streets = (
    build_per_street[build_per_street["count_build"] > 20]
    .dropna()
    .reset_index(drop=True)
)


streets_wth_style = (
    data_gdynia.groupby(["addr_street", "arch_style"], as_index=False)
    .agg(number_of_build_per_style=("osm_id", "count"))
    .sort_values(["addr_street", "number_of_build_per_style"], ascending=(True, False))
)
streets_wth_klasycyzm = (
    streets_wth_style[streets_wth_style["arch_style"] == "klasycyzm"]
    .reset_index(drop=True)
    .drop(columns="arch_style")
)

In [7]:
b_inner = pd.merge(big_streets, streets_wth_klasycyzm, on="addr_street", how="inner")
b_left = pd.merge(big_streets, streets_wth_klasycyzm, on="addr_street", how="left")
b_right = pd.merge(big_streets, streets_wth_klasycyzm, on="addr_street", how="right")
b_outher = pd.merge(big_streets, streets_wth_klasycyzm, on="addr_street", how="outer")

print(f"b_inner_length: {len(b_inner)}")
print(f"b_left_length: {len(b_left)}")
print(f"b_right_length: {len(b_right)}")
print(f"b_outher_length: {len(b_outher)}")

b_inner_length: 9
b_left_length: 12
b_right_length: 12
b_outher_length: 15


In [8]:
assert len(b_outher) == len(b_left) + len(b_right) - len(b_inner)

3.

In [9]:
merge_with_indicator = pd.merge(
    big_streets, streets_wth_klasycyzm, on="addr_street", how="outer", indicator=True
)
merge_with_indicator

,addr_street,count_build,number_of_build_per_style,_merge
0,10 Lutego,48.0,15.0,both
1,3 Maja,26.0,3.0,both
2,Antoniego Abrahama,80.0,5.0,both
3,Antoniego Hryniewickiego,25.0,NaN,left_only
4,Armii Krajowej,36.0,4.0,both
5,Jana z Kolna,41.0,NaN,left_only
6,Jerzego Waszyngtona,29.0,4.0,both
7,Plac Grunwaldzki,NaN,2.0,right_only
8,Plac Kaszubski,NaN,2.0,right_only
9,Starowiejska,59.0,5.0,both


In [10]:
merge_with_indicator["_merge"].value_counts()

_merge
both          9
left_only     3
right_only    3
Name: count, dtype: int64

In [11]:
big_street_build_wthout_style = merge_with_indicator.loc[
    merge_with_indicator["_merge"] == "left_only", ["addr_street"]
]
no_big_street_build_with_style = merge_with_indicator.loc[
    merge_with_indicator["_merge"] == "right_only", ["addr_street"]
]

print(big_street_build_wthout_style)
print(no_big_street_build_with_style)

                 addr_street
3   Antoniego Hryniewickiego
5               Jana z Kolna
10       Stefana Żeromskiego
         addr_street
7   Plac Grunwaldzki
8     Plac Kaszubski
13             Zgoda


4.

In [ ]:
build_per_street = build_per_street.rename(columns={"addr_street": "ulica"})

diff_key_names = pd.merge(
    data_gdynia, build_per_street, left_on="addr_street", right_on="ulica", how="left"
).drop(columns="ulica")

In [13]:
assert diff_key_names.equals(z_left)

5.

In [ ]:
facilities_per_street = data_gdynia_long.groupby("addr_street").agg(
    number=("ground_floor_function", "size")
)
build_per_street = data_gdynia.groupby("addr_street").agg(
    number=("ground_floor_function", "size")
)

num_build_loc = pd.merge(
    build_per_street, facilities_per_street, on="addr_street", how="inner"
)
num_build_loc_suff = pd.merge(
    build_per_street,
    facilities_per_street,
    on="addr_street",
    how="inner",
    suffixes=("_buildings", "_premises"),
)

In [ ]:
assert (num_build_loc["number_x"] <= num_build_loc["number_y"]).all()
assert (
    num_build_loc_suff["number_buildings"] <= num_build_loc_suff["number_premises"]
).all()

6.

In [ ]:
d_merge = pd.merge(
    data_gdynia, build_per_street, left_on="addr_street", right_index=True, how="outer"
)
d_join = data_gdynia.join(build_per_street, on="addr_street", how="outer")


In [17]:
pd.testing.assert_frame_equal(d_merge, d_join)

7.

In [18]:
data_gdynia_styles = data_gdynia.loc[:, ["arch_style"]]
styles_all_func = pd.merge(
    data_gdynia_long,
    data_gdynia_styles,
    right_index=True,
    left_index=True,
    validate="many_to_one",
    how="left",
)
styles_all_func

,addr_street,ground_floor_function,arch_style
id,,,
1,Świętego Piotra,hotel,współczesność
2,Świętego Piotra,mieszkaniowa wielorodzinna,współczesność
3,Węglowa,centrum handlowe,współczesność
4,Świętego Piotra,mieszkaniowa wielorodzinna,współczesność
5,Tadeusza Wendy,centrum medyczne,współczesność
...,...,...,...
1238,Świętojańska,apteka,modernizm
1238,Świętojańska,handel zdrowie,modernizm
1239,Aleja Marszałka Józefa Piłsudskiego,mieszkaniowa wielorodzinna,modernizm


In [19]:
assert len(styles_all_func) == len(data_gdynia_long)

8.

In [20]:
number_of_dist_func = data_gdynia_long.groupby(level=0).agg(
    total_num_of_func=("addr_street", "size")
)
number_of_dist_func = number_of_dist_func.assign(
    total_num_of_func_squared=lambda d: d["total_num_of_func"] ** 2
)

In [21]:
data_gdynia_long_func = data_gdynia_long.loc[:, ["ground_floor_function"]]
many_to_many = pd.merge(
    data_gdynia_long_func,
    data_gdynia_long_func,
    how="inner",
    left_index=True,
    right_index=True,
)

In [22]:
assert len(many_to_many) == number_of_dist_func["total_num_of_func_squared"].sum()

In [23]:
func_combo = many_to_many[
    many_to_many["ground_floor_function_x"] < many_to_many["ground_floor_function_y"]
]
func_combo.groupby(["ground_floor_function_x", "ground_floor_function_y"]).agg(
    "size"
).reset_index(name="num_combo").sort_values("num_combo", ascending=False)

,ground_floor_function_x,ground_floor_function_y,num_combo
31,bank,restauracja,10
160,fryzjer,restauracja,9
367,nieużytek,restauracja,7
308,kawiarnia,restauracja,6
28,bank,kawiarnia,4
...,...,...,...
413,stoważyszenie,usługi masaż,1
414,ubezpieczenia,usługi krawcowa,1
415,usłigi uroda,usługi nieruchomości,1
416,usłigi uroda,żabka,1


In [24]:
len(func_combo)

515

9.

In [ ]:
data_gdynia["date_of_checking"].value_counts()

date_of_checking
2026-09-27    535
2026-09-30    191
2026-09-24    116
2026-09-26    108
2026-09-25     76
2026-09-23     57
2026-09-29     22
Name: count, dtype: int64

In [ ]:
buildings_check_earlier = data_gdynia.loc[
    data_gdynia["date_of_checking"] < "2026-09-27", ["addr_street"]
]
buildings_check_later = data_gdynia.loc[
    data_gdynia["date_of_checking"] >= "2026-09-27", ["addr_street"]
]

pd.concat(
    [buildings_check_earlier, buildings_check_later],
    keys=["earlier_check", "later_check"],
)

addr_street
              id                                       
earlier_check 50                    Stefana Żeromskiego
              51                                Portowa
              55                    Jerzego Waszyngtona
              96                                    NaN
              101                          Jana z Kolna
...                                                 ...
later_check   1237                        Władysława IV
              1238                         Świętojańska
              1239  Aleja Marszałka Józefa Piłsudskiego
              1240  Aleja Marszałka Józefa Piłsudskiego
              1241  Aleja Marszałka Józefa Piłsudskiego

[1105 rows x 1 columns]

In [27]:
assert len(buildings_check_earlier) + len(buildings_check_later) == len(data_gdynia)

In [ ]:
uniq_func_per_street = data_gdynia_long.groupby("addr_street").agg(
    num_unique_func=("ground_floor_function", "nunique")
)

build_per_street = data_gdynia.groupby("addr_street").agg(num_build=("osm_id", "size"))


d_concat = pd.concat([build_per_street, uniq_func_per_street], axis=1)

In [ ]:
d_merge = pd.merge(
    build_per_street,
    uniq_func_per_street,
    left_index=True,
    right_index=True,
    how="outer",
)

In [30]:
assert d_concat.equals(d_merge)

10.

In [ ]:
num_func_buil = data_gdynia_long.groupby(level=0).agg(
    num_func=("ground_floor_function", "count")
)

data_gdynia = pd.merge(
    data_gdynia, num_func_buil, left_index=True, right_index=True, how="outer"
)

In [32]:
assert len(data_gdynia_num_func) == 1105
assert data_gdynia_num_func["num_func"].dtype == "int64"
assert data_gdynia_num_func["num_func"].sum() == 1449
